# binbase Database Maintenance

Low-level PostgreSQL/TimescaleDB maintenance queries for monitoring database health:
size, WAL usage, active connections, cache statistics, autovacuum status, locks, and hypertable info.

> **Requires** a running binbase server at `192.168.100.221:5432`.

In [1]:
from binpan.storage.binbase import connect, close
from binpan.storage.postgresql_database import (
    get_db_size,
    get_table_sizes,
    get_wal_gb_size,
    get_active_connections,
    get_autovacuum_level,
    get_ungranted_locks,
    get_cache_statistics,
    get_hypertable_info,
)

## Connect to binbase

In [2]:
conn, cur = connect()  # password from BINBASE_PASSWORD env var or secret.py

## Database Size

In [3]:
print(f"Database size: {get_db_size(cur)}")

Database size: 5662 MB


## Table Sizes (public schema)

In [4]:
get_table_sizes(cur, only_public_schema=True)

,schemaname,tablename,table_size_kb,indexes_size_kb,total_size_kb
0,public,trades,8.0,32.0,40.0
1,public,agg_trades,8.0,24.0,32.0
2,public,klines,8.0,24.0,32.0
3,public,book_tickers,8.0,16.0,24.0
4,public,tickers,8.0,16.0,24.0
5,public,orderbook_snapshots,8.0,16.0,24.0


## Write-Ahead Log (WAL) Size

In [5]:
wal_gb = get_wal_gb_size(cur)
print(f"WAL size: {wal_gb:.2f} GB")

WAL size: 16.69 GB


## Active Connections

In [6]:
active = get_active_connections(cur, database_name="binbase")
print(f"Active connections to binbase: {active}")

Active connections to binbase: 1


## Autovacuum Status

In [7]:
get_autovacuum_level(cur)

,schemaname,relname,last_autovacuum,last_autoanalyze,n_dead_tup,n_live_tup,n_mod_since_analyze,seq_scan,idx_scan
0,_timescaledb_catalog,continuous_aggs_invalidation_threshold,2026-03-14 21:51:10.433766+00:00,2026-03-14 21:46:50.369412+00:00,41.0,8.0,47.0,1.0,148930.0
1,_timescaledb_catalog,continuous_aggs_hypertable_invalidation_log,2026-03-14 21:39:20.253682+00:00,2026-03-14 21:39:00.247027+00:00,0.0,0.0,13.0,127.0,17795.0
2,_timescaledb_internal,_materialized_hypertable_22,NaT,NaT,0.0,0.0,0.0,4.0,3.0
3,_timescaledb_internal,_hyper_21_15_chunk,NaT,2026-03-14 21:39:20.662082+00:00,0.0,410.0,0.0,3.0,11.0
4,_timescaledb_internal,_compressed_hypertable_11,NaT,NaT,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...
70,_timescaledb_internal,_hyper_15_9_chunk,2026-03-14 21:39:20.437142+00:00,2026-03-14 21:39:20.563471+00:00,0.0,30249.0,80.0,4.0,2824.0
71,_timescaledb_catalog,chunk_rewrite,NaT,NaT,0.0,0.0,0.0,1.0,0.0
72,_timescaledb_internal,_compressed_hypertable_12,NaT,NaT,0.0,0.0,0.0,0.0,NaN
73,_timescaledb_internal,_hyper_26_20_chunk,NaT,NaT,0.0,20.0,20.0,3.0,1.0


## Ungranted Locks

In [8]:
get_ungranted_locks(cur)

""


## Cache Statistics

In [9]:
get_cache_statistics(cur)

,table_name,heap_blks_read,heap_blks_hit,idx_blks_read,idx_blks_hit,toast_blks_read,toast_blks_hit,tidx_blks_read,tidx_blks_hit,heap_cache_hit_rate,idx_cache_hit_rate
0,hypertable,17.0,1894737.0,54.0,1953172.0,NaN,NaN,NaN,NaN,0.999991,0.999972
1,tablespace,0.0,0.0,6.0,151.0,NaN,NaN,NaN,NaN,NaN,0.961783
2,dimension,9.0,799167.0,30.0,833008.0,NaN,NaN,NaN,NaN,0.999989,0.999964
3,dimension_slice,0.0,987247.0,34.0,528820.0,NaN,NaN,NaN,NaN,1.000000,0.999936
4,chunk,0.0,509077.0,114.0,634562.0,NaN,NaN,NaN,NaN,1.000000,0.999820
...,...,...,...,...,...,...,...,...,...,...,...
70,_hyper_24_16_chunk,3.0,209.0,19.0,602.0,0.0,0.0,0.0,0.0,0.985849,0.969404
71,_hyper_23_17_chunk,3.0,312.0,21.0,920.0,0.0,0.0,0.0,0.0,0.990476,0.977683
72,_hyper_22_18_chunk,0.0,103.0,3.0,287.0,0.0,0.0,0.0,0.0,1.000000,0.989655
73,_hyper_25_19_chunk,0.0,20.0,3.0,64.0,0.0,0.0,0.0,0.0,1.000000,0.955224


## Hypertable Information

In [10]:
get_hypertable_info(cur)

,hypertable_schema,hypertable_name,owner,num_dimensions,num_chunks,compression_enabled,tablespaces,primary_dimension,primary_dimension_type
0,public,trades,binbase,1,3,True,None,time,timestamp with time zone
2,public,orderbook_snapshots,binbase,1,2,True,None,time,timestamp with time zone
1,public,agg_trades,binbase,1,0,True,None,time,timestamp with time zone
3,public,klines,binbase,1,0,True,None,time,timestamp with time zone
4,public,book_tickers,binbase,1,0,True,None,time,timestamp with time zone
5,public,tickers,binbase,1,0,True,None,time,timestamp with time zone


## Close Connection

In [11]:
close(conn)
print("Connection closed.")

Connection closed.
